# 03-01 Transformer 架构原理

**面试必考！** 做 AI Agent 开发必须能解释底层模型的工作原理。

**本节目标**：
- 理解 Transformer 整体架构（Encoder-Decoder / Decoder-only）
- 手推 Self-Attention 计算流程（Q/K/V）
- 理解 Multi-Head Attention、Position Encoding、FFN
- 了解 GPT 系列（Decoder-only）的推理过程
- 掌握面试中关于 KV Cache、位置编码的常见问题

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['DejaVu Sans']

# 设置随机种子
np.random.seed(42)

## 1. Self-Attention 原理

Attention 的核心思想：**句子中每个词关注其他词的程度**

公式：
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q (Query)**: 当前词想要查询什么
- **K (Key)**: 每个词提供的索引信息
- **V (Value)**: 每个词实际携带的信息
- **√d_k**: 缩放因子，防止点积过大导致梯度消失

In [ ]:
def softmax(x: np.ndarray) -> np.ndarray:
    e_x = np.exp(x - x.max(axis=-1, keepdims=True))  # 数值稳定
    return e_x / e_x.sum(axis=-1, keepdims=True)

def self_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray, 
                   mask: np.ndarray = None) -> tuple:
    """
    Self-Attention 的 NumPy 实现
    
    Args:
        Q: Query 矩阵 [seq_len, d_k]
        K: Key 矩阵   [seq_len, d_k]
        V: Value 矩阵 [seq_len, d_v]
        mask: 因果掩码（Decoder 用），防止看到未来信息
    
    Returns:
        output: 注意力输出 [seq_len, d_v]
        weights: 注意力权重 [seq_len, seq_len]
    """
    d_k = Q.shape[-1]
    
    # Step 1: 计算注意力分数 (Q · K^T) / sqrt(d_k)
    scores = Q @ K.T / np.sqrt(d_k)   # [seq_len, seq_len]
    
    # Step 2: 应用因果掩码（Decoder-only 模型用，如 GPT）
    if mask is not None:
        scores = scores + mask * -1e9  # 被遮掩的位置设为 -inf
    
    # Step 3: Softmax 归一化
    weights = softmax(scores)          # [seq_len, seq_len]
    
    # Step 4: 加权求和 Values
    output = weights @ V               # [seq_len, d_v]
    
    return output, weights

# 示例：处理 4 个词，d_k=8
seq_len, d_k = 4, 8
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

output, weights = self_attention(Q, K, V)
print(f"输入 Q shape: {Q.shape}")
print(f"注意力权重 shape: {weights.shape}")
print(f"输出 shape: {output.shape}")
print(f"\n注意力权重（每行之和=1）:\n{np.round(weights, 3)}")

In [ ]:
# 可视化注意力权重
tokens = ["B站", "广告", "点击", "率"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Encoder Self-Attention（双向）
_, enc_weights = self_attention(Q, K, V, mask=None)
im1 = ax1.imshow(enc_weights, cmap='Blues', vmin=0, vmax=1)
ax1.set_xticks(range(len(tokens))); ax1.set_xticklabels(tokens)
ax1.set_yticks(range(len(tokens))); ax1.set_yticklabels(tokens)
ax1.set_title("Encoder Self-Attention\n（每个词可关注所有词）")
plt.colorbar(im1, ax=ax1)

# Decoder Causal Attention（单向，只看过去）
causal_mask = np.triu(np.ones((seq_len, seq_len)), k=1)  # 上三角为1（遮掩未来）
_, dec_weights = self_attention(Q, K, V, mask=causal_mask)
im2 = ax2.imshow(dec_weights, cmap='Blues', vmin=0, vmax=1)
ax2.set_xticks(range(len(tokens))); ax2.set_xticklabels(tokens)
ax2.set_yticks(range(len(tokens))); ax2.set_yticklabels(tokens)
ax2.set_title("Decoder Causal Attention\n（只能看过去，GPT 使用此方式）")
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.savefig("attention_visualization.png", dpi=150, bbox_inches='tight')
plt.show()
print("注意左图（双向）vs 右图（因果掩码/单向）的区别")

## 2. Multi-Head Attention

多头注意力 = 多个 Self-Attention 并行，让模型从不同角度（子空间）理解句子

$$\text{MultiHead}(Q,K,V) = \text{Concat}(head_1,...,head_h)W^O$$
$$\text{where} \; head_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

In [ ]:
def multi_head_attention(X: np.ndarray, num_heads: int, d_model: int) -> np.ndarray:
    """
    Multi-Head Self-Attention
    
    Args:
        X: 输入 [seq_len, d_model]
        num_heads: 注意力头数
        d_model: 模型维度
    """
    d_head = d_model // num_heads
    seq_len = X.shape[0]
    
    # 为每个头创建独立的投影矩阵
    heads_output = []
    for h in range(num_heads):
        # 每个头有自己的 WQ, WK, WV
        W_q = np.random.randn(d_model, d_head) * 0.1
        W_k = np.random.randn(d_model, d_head) * 0.1
        W_v = np.random.randn(d_model, d_head) * 0.1
        
        Q_h = X @ W_q  # [seq_len, d_head]
        K_h = X @ W_k  # [seq_len, d_head]
        V_h = X @ W_v  # [seq_len, d_head]
        
        head_out, _ = self_attention(Q_h, K_h, V_h)
        heads_output.append(head_out)
    
    # 拼接所有头的输出
    concat = np.concatenate(heads_output, axis=-1)  # [seq_len, d_model]
    
    # 最终投影
    W_o = np.random.randn(d_model, d_model) * 0.1
    return concat @ W_o  # [seq_len, d_model]

# GPT-2 small: 12 heads, d_model=768
X = np.random.randn(10, 128)  # 10个词, d_model=128
output = multi_head_attention(X, num_heads=8, d_model=128)
print(f"输入: {X.shape} → 输出: {output.shape}")
print("维度不变，但每个位置的表示融合了其他位置的信息")

## 3. 面试高频问题

### Q1: Self-Attention 的时间/空间复杂度是多少？

In [ ]:
# Attention 的复杂度：O(n² · d)
# n: 序列长度, d: 模型维度
# 这就是为什么 context window 很贵！

seq_lengths = [512, 1024, 2048, 4096, 8192, 128000]  # GPT-4o 支持 128K
d_model = 768

print(f"{'序列长度':>10} {'Attention 矩阵大小':>20} {'内存 (FP16, MB)':>18}")
print("-" * 52)
for n in seq_lengths:
    matrix_size = n * n  # Q·K^T 矩阵 shape: [n, n]
    memory_mb = matrix_size * 2 / 1024**2  # FP16 = 2 bytes
    print(f"{n:>10,} {matrix_size:>20,} {memory_mb:>18.1f}")
    
print("\n这就是 FlashAttention、GQA 等优化技术存在的原因！")

In [ ]:
# Q2: KV Cache 是什么？为什么能加速推理？
print("""
KV Cache 原理：

LLM 生成 token 是自回归的：
  "B站" → "B站广" → "B站广告" → "B站广告点" → ...

没有 KV Cache：
  每次生成新 token，都要重新计算所有历史 token 的 K, V
  第 n 步：O(n²) 计算量
  总计：O(n³)

有 KV Cache：
  历史 token 的 K, V 已计算并存在 GPU 显存中
  每步只计算新 token 的 Q，然后和缓存的 K, V 做 Attention
  第 n 步：O(n) 计算量
  总计：O(n²)  ← 速度大幅提升

代价：显存占用 = 2 × n_layers × n_heads × seq_len × d_head × bytes
  → context window 越长，KV Cache 越大
""")

# Q3: 为什么 GPT 用 Decoder-only，而不是 Encoder-Decoder？
print("""
Decoder-only (GPT) vs Encoder-Decoder (T5/BART):

Decoder-only 优势：
  - 训练目标简单：预测下一个 token（Next Token Prediction）
  - 规模效应更好：同等参数量，生成效果更强
  - 统一架构：理解和生成都用同一个模型
  - In-context learning 能力更强

Encoder-Decoder 优势：
  - 理解任务（分类/摘要）时 Encoder 可双向关注全文
  - 输入输出分离，适合翻译、摘要

LLM 时代的趋势：Decoder-only 统一了大多数任务
""")

## 4. 位置编码（Position Encoding）

In [ ]:
# 原始 Sinusoidal 位置编码（Attention is All You Need）
def sinusoidal_encoding(seq_len: int, d_model: int) -> np.ndarray:
    PE = np.zeros((seq_len, d_model))
    pos = np.arange(seq_len)[:, np.newaxis]      # [seq_len, 1]
    div = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
    
    PE[:, 0::2] = np.sin(pos * div)  # 偶数维度
    PE[:, 1::2] = np.cos(pos * div)  # 奇数维度
    return PE

PE = sinusoidal_encoding(50, 64)

plt.figure(figsize=(10, 4))
plt.imshow(PE.T, aspect='auto', cmap='RdBu')
plt.xlabel('Token 位置')
plt.ylabel('维度')
plt.title('Sinusoidal 位置编码 — 不同频率的正弦/余弦波')
plt.colorbar()
plt.tight_layout()
plt.show()

print("""
面试要点 —— 位置编码的演进：
1. 绝对位置编码（Sinusoidal）: 原始 Transformer，外推性差
2. 可学习位置编码（Learned PE）: BERT/GPT-2，有最大序列长度限制
3. 相对位置编码（ALiBi, T5 RPE）: 更好的长度外推
4. RoPE（Rotary Position Embedding）: LLaMA/GPT-NeoX/现代 LLM 主流
   - 将位置信息编码为旋转矩阵，天然支持相对位置关系
   - 通过位置内插（PI）和 YaRN 可扩展 context 长度
""")

## 5. 完整 Transformer Block 概览

In [ ]:
print("""
GPT-style Transformer Block（Decoder-only）：

输入 x
  ↓
  x = x + LayerNorm(CausalSelfAttention(x))   # Pre-norm（现代 LLM 常用）
  ↓
  x = x + LayerNorm(FFN(x))
  ↓
输出 x

FFN（Feed-Forward Network）：
  FFN(x) = GELU(x · W1 + b1) · W2 + b2
  维度：d_model → 4*d_model → d_model
  作用：引入非线性，增强模型表达能力

关键超参数（GPT-3 175B 为例）：
  n_layers: 96 层
  d_model: 12288
  n_heads: 96
  d_head: 128 (= d_model / n_heads)
  context: 2048 tokens
""")

## 面试速记卡

| 问题 | 要点 |
|------|------|
| Attention 复杂度 | 时间 O(n²d)，空间 O(n²)，n 是序列长度 |
| 为什么 scale √d_k | 点积过大→softmax 趋于 one-hot→梯度消失 |
| KV Cache 作用 | 缓存历史 K,V，推理由 O(n³)→O(n²) |
| RoPE 优势 | 相对位置，外推性好，旋转编码保持内积性质 |
| LayerNorm 位置 | Pre-norm（现代主流）vs Post-norm（原始论文）|
| 幻觉根本原因 | 训练目标是下一 token 概率最大化，不保证事实准确 |

**下一节**: `02_tokenization_embedding.ipynb`